In [1]:
# [CELL 1] STUDENT CONFIGURATION
from pathlib import Path

LAST_NAME = "Papa"
STUDENT_ID = "TUPM-26-1927"

SEED_DIGIT = int(STUDENT_ID[-1])
ID_SUM = sum(int(d) for d in STUDENT_ID if d.isdigit())

TARGET_OS = "Android" if SEED_DIGIT % 2 == 0 else "iOS"
TARGET_CLASS = (SEED_DIGIT % 5) + 1

DATA_DIR = Path("data")
OUTPUT_DIR = Path("output")
BACKUP_DIR = Path("backup")
OUTPUT_DIR.mkdir(exist_ok=True)
BACKUP_DIR.mkdir(exist_ok=True)

print("Seed:", SEED_DIGIT)
print("ID Sum:", ID_SUM)
print("Target operating system:", TARGET_OS)
print("Target behavior class:", TARGET_CLASS)
print("Workspace:", Path(".").resolve())

Seed: 7
ID Sum: 27
Target operating system: iOS
Target behavior class: 3
Workspace: C:\Users\Cyrus\OneDrive\Desktop\Lab6_File_Handling\Lab6_File_Handling_Papa\Lab6_File_Handling_Additional_Exercises


In [2]:
# [CELL 2] INSPECT THE DATASET
import csv

dataset_path = DATA_DIR / "user_behavior_dataset.csv"
print("Exists:", dataset_path.exists())

with open(dataset_path, "r", encoding="utf-8", newline="") as file:
    reader = csv.reader(file)
    header = next(reader)
    print("Columns:")
    for index, name in enumerate(header):
        print(f"  {index}: {name!r}")
    print("First row:", next(reader))

Exists: True
Columns:
  0: 'User ID'
  1: 'Device Model'
  2: 'Operating System'
  3: 'App Usage Time (min/day)'
  4: 'Screen On Time (hours/day)'
  5: 'Battery Drain (mAh/day)'
  6: 'Number of Apps Installed'
  7: 'Data Usage (MB/day)'
  8: 'Age'
  9: 'Gender'
  10: 'User Behavior Class'
First row: ['1', 'Google Pixel 5', 'Android', '393', '6.4', '1872', '67', '1122', '40', 'Male', '4']


In [3]:
# [CELL 3] ANALYZE, EXPORT, SUMMARIZE, AND BACK UP
import json
import shutil

EXPORT_FIELDS = [
    "User ID",
    "Device Model",
    "Operating System",
    "Battery Drain (mAh/day)",
    "Data Usage (MB/day)",
]
REQUIRED_FIELDS = EXPORT_FIELDS + ["User Behavior Class"]

total_records = 0
os_matches = 0
invalid_records = 0
selected = []
battery_total = 0.0
data_total = 0.0

with open(dataset_path, "r", encoding="utf-8", newline="") as file:
    reader = csv.DictReader(file)

    missing = [name for name in REQUIRED_FIELDS if name not in reader.fieldnames]
    if missing:
        raise KeyError(f"Missing columns: {missing}. Found: {reader.fieldnames}")

    for row in reader:
        total_records += 1

        if row["Operating System"].strip() != TARGET_OS:
            continue
        os_matches += 1

        try:
            behavior = int(row["User Behavior Class"])
            battery = float(row["Battery Drain (mAh/day)"])
            data_usage = float(row["Data Usage (MB/day)"])
        except ValueError:
            invalid_records += 1
            continue

        if behavior != TARGET_CLASS:
            continue

        selected.append(row)
        battery_total += battery
        data_total += data_usage

selected_count = len(selected)
avg_battery = battery_total / selected_count if selected_count else 0.0
avg_data = data_total / selected_count if selected_count else 0.0

# Export the selected users
exported_csv_path = OUTPUT_DIR / f"{LAST_NAME}_{STUDENT_ID}_mobile_selected.csv"
with open(exported_csv_path, "w", encoding="utf-8", newline="") as file:
    writer = csv.DictWriter(file, fieldnames=EXPORT_FIELDS, extrasaction="ignore")
    writer.writeheader()
    writer.writerows(selected)

# Verify the exported row count
with open(exported_csv_path, "r", encoding="utf-8", newline="") as file:
    exported_rows = sum(1 for _ in csv.reader(file)) - 1  # minus the header
count_verified = exported_rows == selected_count

# Write the JSON summary
summary_json_path = OUTPUT_DIR / f"{LAST_NAME}_{STUDENT_ID}_mobile_summary.json"
summary = {
    "last_name": LAST_NAME,
    "student_id": STUDENT_ID,
    "seed_digit": SEED_DIGIT,
    "target_operating_system": TARGET_OS,
    "target_behavior_class": TARGET_CLASS,
    "total_records": total_records,
    "records_matching_os": os_matches,
    "invalid_records": invalid_records,
    "records_in_target_class": selected_count,
    "average_battery_drain_mah_per_day": round(avg_battery, 3),
    "average_data_usage_mb_per_day": round(avg_data, 3),
    "exported_row_count": exported_rows,
    "count_verified": count_verified,
    "note": "" if selected_count else "No users matched the selected OS and behavior class.",
}
with open(summary_json_path, "w", encoding="utf-8") as file:
    json.dump(summary, file, indent=4)

# Back up the exported CSV
backup_path = BACKUP_DIR / exported_csv_path.name
shutil.copy2(exported_csv_path, backup_path)

print("Operating system and behavior class:", TARGET_OS, "| class", TARGET_CLASS)
print("Total records in dataset:", total_records)
print("Records matching operating system:", os_matches)
print("Records in target behavior class:", selected_count)
print("Invalid records skipped:", invalid_records)
print("Average battery drain (mAh/day):", round(avg_battery, 3))
print("Average data usage (MB/day):", round(avg_data, 3))
print("Exported rows:", exported_rows, "| Selected:", selected_count)
print("Count verified:", count_verified)
print("Exported CSV:", exported_csv_path)
print("JSON summary:", summary_json_path)
print("Backup file:", backup_path)
print("Backup size:", backup_path.stat().st_size, "bytes")

Operating system and behavior class: iOS | class 3
Total records in dataset: 700
Records matching operating system: 146
Records in target behavior class: 32
Invalid records skipped: 0
Average battery drain (mAh/day): 1492.031
Average data usage (MB/day): 828.094
Exported rows: 32 | Selected: 32
Count verified: True
Exported CSV: output\Papa_TUPM-26-1927_mobile_selected.csv
JSON summary: output\Papa_TUPM-26-1927_mobile_summary.json
Backup file: backup\Papa_TUPM-26-1927_mobile_selected.csv
Backup size: 973 bytes
